# 06 — Scientific Abstract GPT Evaluation

This notebook evaluates:

- Test loss
- Test perplexity
- Train-validation gap
- Structural-token compliance
- Generated abstract length
- Word diversity
- Trigram repetition
- Empty-output rate

The evaluation report is saved for the final Gradio demo.

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [2]:
import os
import sys
import re
import json
import math
from collections import Counter

import numpy as np
import pandas as pd
import torch

In [3]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

TOKEN_STREAM_FOLDER = os.path.join(
    PROJECT_PATH,
    "data",
    "tokenized_bpe_streams"
)

TEST_STREAM_PATH = os.path.join(
    TOKEN_STREAM_FOLDER,
    "test_tokens.bin"
)

STREAM_METADATA_PATH = os.path.join(
    TOKEN_STREAM_FOLDER,
    "stream_metadata.json"
)

TOKENIZER_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "bpe_tokenizer",
    "tokenizer.json"
)

BEST_CHECKPOINT_PATH = os.path.join(
    PROJECT_PATH,
    "models",
    "best_model.pt"
)

TRAINING_HISTORY_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "training_history.json"
)

GENERATED_SAMPLES_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "generated_samples.json"
)

EVALUATION_REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "evaluation_report.json"
)

EVALUATION_TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "evaluation_samples.csv"
)

if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project path:",
    PROJECT_PATH
)

Project path: /content/drive/MyDrive/Scientific-Abstract-GPT


In [4]:
required_files = {
    "best model checkpoint": (
        BEST_CHECKPOINT_PATH
    ),
    "test token stream": (
        TEST_STREAM_PATH
    ),
    "stream metadata": (
        STREAM_METADATA_PATH
    ),
    "tokenizer": (
        TOKENIZER_PATH
    ),
    "training history": (
        TRAINING_HISTORY_PATH
    ),
    "generated samples": (
        GENERATED_SAMPLES_PATH
    )
}

missing_files = []

for file_name, file_path in (
    required_files.items()
):

    exists = os.path.exists(
        file_path
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:

        missing_files.append(
            file_path
        )

if missing_files:

    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print(
    "\nAll required files are available."
)

best model checkpoint: FOUND
test token stream: FOUND
stream metadata: FOUND
tokenizer: FOUND
training history: FOUND
generated samples: FOUND

All required files are available.


Import GPT Components and Select Device

In [5]:
from src.gpt_components import (
    GPTConfig,
    GPTLanguageModel,
    load_tokenizer,
    set_seed
)

SEED = 42

set_seed(
    SEED
)

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    device.type == "cuda"
)

print(
    "Selected device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Selected device: cuda:0
GPU: Tesla T4


Load Tokenizer and Best Model

In [6]:
tokenizer = load_tokenizer(
    TOKENIZER_PATH
)

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_config = GPTConfig.from_dict(
    checkpoint[
        "model_config"
    ]
)

if (
    tokenizer.get_vocab_size()
    != model_config.vocab_size
):

    raise ValueError(
        "Tokenizer vocabulary size does not "
        "match the model vocabulary size."
    )

model = GPTLanguageModel(
    model_config
).to(device)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print(
    "Best checkpoint loaded successfully."
)

print(
    "Checkpoint step:",
    checkpoint["step"]
)

print(
    "Validation loss:",
    f"{checkpoint['validation_loss']:.4f}"
)

print(
    "Model device:",
    next(model.parameters()).device
)

Best checkpoint loaded successfully.
Checkpoint step: 3000
Validation loss: 3.8935
Model device: cuda:0


In [7]:
with open(
    STREAM_METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    stream_metadata = json.load(
        file
    )

stream_dtype = np.dtype(
    stream_metadata.get(
        "dtype",
        "uint16"
    )
)

test_tokens = np.memmap(
    TEST_STREAM_PATH,
    dtype=stream_dtype,
    mode="r"
)

if (
    len(test_tokens)
    <= model_config.block_size + 1
):

    raise ValueError(
        "Test token stream is too short."
    )

if (
    int(np.max(test_tokens))
    >= model_config.vocab_size
):

    raise ValueError(
        "Test stream contains an invalid token ID."
    )

print(
    "Test tokens:",
    f"{len(test_tokens):,}"
)

print(
    "Stream dtype:",
    stream_dtype
)

print(
    "Test stream validation passed."
)

Test tokens: 2,514,500
Stream dtype: uint16
Test stream validation passed.


Create Test Batches

In [8]:
EVALUATION_BATCH_SIZE = 16
EVALUATION_BATCHES = 100


def get_test_batch():

    maximum_start_index = (
        len(test_tokens)
        - model_config.block_size
        - 1
    )

    start_indices = torch.randint(
        low=0,
        high=maximum_start_index,
        size=(
            EVALUATION_BATCH_SIZE,
        )
    )

    input_sequences = []
    target_sequences = []

    for start_index in (
        start_indices.tolist()
    ):

        input_array = np.asarray(
            test_tokens[
                start_index:
                start_index
                + model_config.block_size
            ],
            dtype=np.int64
        ).copy()

        target_array = np.asarray(
            test_tokens[
                start_index + 1:
                start_index
                + model_config.block_size
                + 1
            ],
            dtype=np.int64
        ).copy()

        input_sequences.append(
            torch.from_numpy(
                input_array
            )
        )

        target_sequences.append(
            torch.from_numpy(
                target_array
            )
        )

    input_batch = torch.stack(
        input_sequences
    ).to(
        device,
        non_blocking=True
    )

    target_batch = torch.stack(
        target_sequences
    ).to(
        device,
        non_blocking=True
    )

    return (
        input_batch,
        target_batch
    )


test_inputs, test_targets = (
    get_test_batch()
)

print(
    "Input shape:",
    test_inputs.shape
)

print(
    "Target shape:",
    test_targets.shape
)

print(
    "Input device:",
    test_inputs.device
)

Input shape: torch.Size([16, 256])
Target shape: torch.Size([16, 256])
Input device: cuda:0


Calculate Test Loss and Perplexity

In [9]:
@torch.no_grad()
def calculate_test_loss():

    model.eval()

    losses = []

    for _ in range(
        EVALUATION_BATCHES
    ):

        inputs, targets = (
            get_test_batch()
        )

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            _, loss = model(
                inputs,
                targets
            )

        losses.append(
            loss.detach().float().cpu().item()
        )

    return float(
        np.mean(losses)
    )


test_loss = calculate_test_loss()

test_perplexity = math.exp(
    min(
        test_loss,
        20
    )
)

print(
    "Test loss:",
    f"{test_loss:.4f}"
)

print(
    "Test perplexity:",
    f"{test_perplexity:.4f}"
)

Test loss: 3.8807
Test perplexity: 48.4577


Load Training Results and Generated Samples

In [10]:
with open(
    TRAINING_HISTORY_PATH,
    "r",
    encoding="utf-8"
) as file:

    training_results = json.load(
        file
    )

with open(
    GENERATED_SAMPLES_PATH,
    "r",
    encoding="utf-8"
) as file:

    generation_results = json.load(
        file
    )

generated_samples = (
    generation_results[
        "samples"
    ]
)

final_train_loss = float(
    training_results[
        "final_train_loss"
    ]
)

final_validation_loss = float(
    training_results[
        "final_validation_loss"
    ]
)

train_validation_gap = abs(
    final_validation_loss
    - final_train_loss
)

print(
    "Final training loss:",
    f"{final_train_loss:.4f}"
)

print(
    "Final validation loss:",
    f"{final_validation_loss:.4f}"
)

print(
    "Train-validation gap:",
    f"{train_validation_gap:.4f}"
)

print(
    "Generated samples:",
    len(generated_samples)
)

Final training loss: 3.8820
Final validation loss: 3.8883
Train-validation gap: 0.0062
Generated samples: 6


Define Generated-Text Metrics

In [11]:
STRUCTURAL_TOKENS = [
    "<TITLE>",
    "<SUBJECT>",
    "<ABSTRACT>",
    "<END>"
]


def tokenize_words(
    text
):

    return re.findall(
        r"[A-Za-z]+(?:'[A-Za-z]+)?",
        text.lower()
    )


def calculate_word_diversity(
    words
):

    if not words:

        return 0.0

    return (
        len(set(words))
        / len(words)
    )


def calculate_trigram_repetition(
    words
):

    if len(words) < 3:

        return 0.0

    trigrams = [
        tuple(
            words[
                index:
                index + 3
            ]
        )
        for index in range(
            len(words) - 2
        )
    ]

    trigram_counts = Counter(
        trigrams
    )

    repeated_occurrences = sum(
        count - 1
        for count in (
            trigram_counts.values()
        )
        if count > 1
    )

    return (
        repeated_occurrences
        / len(trigrams)
    )


def calculate_structural_compliance(
    generated_text
):

    positions = [
        generated_text.find(
            token
        )
        for token in STRUCTURAL_TOKENS
    ]

    tokens_present = [
        position >= 0
        for position in positions
    ]

    present_score = (
        sum(tokens_present)
        / len(STRUCTURAL_TOKENS)
    )

    available_positions = [
        position
        for position in positions
        if position >= 0
    ]

    correct_order = (
        available_positions
        == sorted(
            available_positions
        )
    )

    if not correct_order:

        return 0.0

    return present_score

In [12]:
sample_evaluations = []

for sample_number, sample in enumerate(
    generated_samples,
    start=1
):

    abstract = sample.get(
        "abstract",
        ""
    ).strip()

    generated_text = sample.get(
        "generated_text",
        ""
    )

    words = tokenize_words(
        abstract
    )

    structural_compliance = (
        calculate_structural_compliance(
            generated_text
        )
    )

    sample_evaluation = {
        "sample_number": (
            sample_number
        ),
        "title": sample.get(
            "title",
            ""
        ),
        "subject": sample.get(
            "subject",
            ""
        ),
        "word_count": (
            len(words)
        ),
        "unique_word_count": (
            len(set(words))
        ),
        "word_diversity": (
            calculate_word_diversity(
                words
            )
        ),
        "trigram_repetition_rate": (
            calculate_trigram_repetition(
                words
            )
        ),
        "structural_compliance": (
            structural_compliance
        ),
        "contains_end_token": (
            "<END>"
            in generated_text
        ),
        "is_empty": (
            len(words) == 0
        )
    }

    sample_evaluations.append(
        sample_evaluation
    )

evaluation_dataframe = pd.DataFrame(
    sample_evaluations
)

evaluation_dataframe

,sample_number,title,subject,word_count,unique_word_count,word_diversity,trigram_repetition_rate,structural_compliance,contains_end_token,is_empty
0,1,Deep Learning for Medical Image Classification,Machine Learning,139,88,0.633094,0.007299,1.00,True,False
1,2,Transformer Models for Scientific Document Sum...,Computation and Language,101,77,0.762376,0.000000,1.00,True,False
2,3,Reinforcement Learning for Autonomous Robot Na...,Artificial Intelligence,103,60,0.582524,0.019802,1.00,True,False
3,4,Neural Networks for Natural Language Understan...,Computation and Language,177,109,0.615819,0.022857,0.75,False,False
4,5,Explainable Artificial Intelligence for Health...,Artificial Intelligence,170,109,0.641176,0.000000,1.00,True,False
5,6,Self-Supervised Representation Learning from U...,Machine Learning,92,72,0.782609,0.000000,1.00,True,False


Calculate Overall Generation Metrics

In [13]:
average_abstract_words = float(
    evaluation_dataframe[
        "word_count"
    ].mean()
)

average_word_diversity = float(
    evaluation_dataframe[
        "word_diversity"
    ].mean()
)

average_trigram_repetition = float(
    evaluation_dataframe[
        "trigram_repetition_rate"
    ].mean()
)

structural_compliance_rate = float(
    evaluation_dataframe[
        "structural_compliance"
    ].mean()
)

end_token_rate = float(
    evaluation_dataframe[
        "contains_end_token"
    ].mean()
)

empty_output_rate = float(
    evaluation_dataframe[
        "is_empty"
    ].mean()
)

print(
    "Average generated abstract words:",
    f"{average_abstract_words:.2f}"
)

print(
    "Average word diversity:",
    f"{average_word_diversity:.4f}"
)

print(
    "Average trigram repetition rate:",
    f"{average_trigram_repetition:.4f}"
)

print(
    "Structural compliance:",
    f"{structural_compliance_rate * 100:.2f}%"
)

print(
    "END-token completion rate:",
    f"{end_token_rate * 100:.2f}%"
)

print(
    "Empty-output rate:",
    f"{empty_output_rate * 100:.2f}%"
)

Average generated abstract words: 130.33
Average word diversity: 0.6696
Average trigram repetition rate: 0.0083
Structural compliance: 95.83%
END-token completion rate: 83.33%
Empty-output rate: 0.00%


In [14]:
evaluation_report = {
    "checkpoint": {
        "path": (
            BEST_CHECKPOINT_PATH
        ),
        "step": int(
            checkpoint["step"]
        ),
        "validation_loss_at_save": float(
            checkpoint[
                "validation_loss"
            ]
        )
    },
    "language_model_metrics": {
        "final_training_loss": (
            final_train_loss
        ),
        "final_validation_loss": (
            final_validation_loss
        ),
        "train_validation_gap": (
            train_validation_gap
        ),
        "test_loss": (
            test_loss
        ),
        "test_perplexity": (
            test_perplexity
        )
    },
    "generation_metrics": {
        "number_of_samples": int(
            len(
                evaluation_dataframe
            )
        ),
        "average_abstract_words": (
            average_abstract_words
        ),
        "average_word_diversity": (
            average_word_diversity
        ),
        "average_trigram_repetition_rate": (
            average_trigram_repetition
        ),
        "structural_compliance_rate": (
            structural_compliance_rate
        ),
        "end_token_completion_rate": (
            end_token_rate
        ),
        "empty_output_rate": (
            empty_output_rate
        )
    },
    "model_config": (
        model_config.to_dict()
    ),
    "sample_evaluations": (
        sample_evaluations
    )
}

Save Evaluation Outputs

In [15]:
with open(
    EVALUATION_REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        evaluation_report,
        file,
        indent=2,
        ensure_ascii=False
    )

evaluation_dataframe.to_csv(
    EVALUATION_TABLE_PATH,
    index=False
)

print(
    "Evaluation report saved:"
)

print(
    EVALUATION_REPORT_PATH
)

print(
    "\nEvaluation table saved:"
)

print(
    EVALUATION_TABLE_PATH
)

Evaluation report saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/evaluation_report.json

Evaluation table saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/evaluation_samples.csv


In [16]:
evaluation_outputs = {
    "evaluation report": (
        EVALUATION_REPORT_PATH
    ),
    "evaluation sample table": (
        EVALUATION_TABLE_PATH
    )
}

all_outputs_found = True

for output_name, output_path in (
    evaluation_outputs.items()
):

    exists = os.path.exists(
        output_path
    )

    print(
        f"{output_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    all_outputs_found = (
        all_outputs_found
        and exists
    )

if not all_outputs_found:

    raise RuntimeError(
        "Evaluation output files "
        "were not created."
    )

with open(
    EVALUATION_REPORT_PATH,
    "r",
    encoding="utf-8"
) as file:

    saved_report = json.load(
        file
    )

print(
    "\nSaved test loss:",
    f"{saved_report['language_model_metrics']['test_loss']:.4f}"
)

print(
    "Saved test perplexity:",
    f"{saved_report['language_model_metrics']['test_perplexity']:.4f}"
)

evaluation report: FOUND
evaluation sample table: FOUND

Saved test loss: 3.8807
Saved test perplexity: 48.4577


In [17]:
print(
    "=" * 70
)

print(
    "FINAL EVALUATION SUMMARY"
)

print(
    "=" * 70
)

print(
    "Final training loss:",
    f"{final_train_loss:.4f}"
)

print(
    "Final validation loss:",
    f"{final_validation_loss:.4f}"
)

print(
    "Train-validation gap:",
    f"{train_validation_gap:.4f}"
)

print(
    "Test loss:",
    f"{test_loss:.4f}"
)

print(
    "Test perplexity:",
    f"{test_perplexity:.4f}"
)

print(
    "Structural compliance:",
    f"{structural_compliance_rate * 100:.2f}%"
)

print(
    "END-token completion rate:",
    f"{end_token_rate * 100:.2f}%"
)

print(
    "Average generated abstract words:",
    f"{average_abstract_words:.2f}"
)

print(
    "Average word diversity:",
    f"{average_word_diversity:.4f}"
)

print(
    "Average trigram repetition rate:",
    f"{average_trigram_repetition:.4f}"
)

print(
    "Empty-output rate:",
    f"{empty_output_rate * 100:.2f}%"
)

print(
    "\nEvaluation report:"
)

print(
    EVALUATION_REPORT_PATH
)


FINAL EVALUATION SUMMARY
Final training loss: 3.8820
Final validation loss: 3.8883
Train-validation gap: 0.0062
Test loss: 3.8807
Test perplexity: 48.4577
Structural compliance: 95.83%
END-token completion rate: 83.33%
Average generated abstract words: 130.33
Average word diversity: 0.6696
Average trigram repetition rate: 0.0083
Empty-output rate: 0.00%

Evaluation report:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/evaluation_report.json
